In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import hashlib
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs("reports/shap", exist_ok=True)

print("=" * 60)
print("  EXPLICABILITÉ DES DÉCISIONS — APPROXIMATION SHAP")
print("=" * 60)

  EXPLICABILITÉ SHAP RÉELLE — TreeExplainer


In [ ]:
# ─────────────────────────────────────────────────────────
# CHARGEMENT
# ─────────────────────────────────────────────────────────
print("\n[1/5] Chargement des données et du modèle...")

# Dataset engineeré si disponible
DATA_PATH = ("data/transactions_engineered.csv"
             if os.path.exists("data/transactions_engineered.csv")
             else "data/transactions_bancaires.csv")
df = pd.read_csv(DATA_PATH)

NUMERIC_BASE = [
    "heure", "jour_semaine", "est_weekend",
    "montant_mad", "est_etranger",
    "delta_km", "delta_min_last_tx", "nb_tx_1h",
    "est_nouveau_device", "age_client", "age_compte_jours",
    "ratio_montant_moy", "risque_horaire",
]
ENGINEERED = [f for f in [
    "log_montant_mad", "log_delta_km", "est_nuit",
    "vitesse_km_min", "montant_x_etranger", "risque_x_nb_tx",
] if f in df.columns]

NUMERIC_FEATURES = NUMERIC_BASE + ENGINEERED
CATEGORICAL_FEATURES = ["type_transaction", "device_type", "segment_revenu", "type_carte"]

df_model = df.copy()
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_model[col + "_enc"] = le.fit_transform(df_model[col].astype(str))

ENCODED_CAT  = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENCODED_CAT

FEATURE_LABELS = {
    "delta_km":             "Distance depuis dernière tx (km)",
    "montant_mad":          "Montant de la transaction (MAD)",
    "ratio_montant_moy":    "Ratio montant / moyenne client",
    "nb_tx_1h":             "Nombre de tx dans la dernière heure",
    "heure":                "Heure de la transaction",
    "delta_min_last_tx":    "Délai depuis dernière tx (min)",
    "est_etranger":         "Transaction à l'étranger (0/1)",
    "risque_horaire":       "Niveau de risque horaire (0-3)",
    "est_nouveau_device":   "Nouveau device (0/1)",
    "age_compte_jours":     "Ancienneté du compte (jours)",
    "age_client":           "Âge du client",
    "est_weekend":          "Transaction le week-end (0/1)",
    "type_transaction_enc": "Type de transaction",
    "device_type_enc":      "Type de device",
    "segment_revenu_enc":   "Segment de revenu",
    "type_carte_enc":       "Type de carte",
    "jour_semaine":         "Jour de la semaine",
    # ── Features engineerées (EDA v2.0) ──────────────────
    "log_montant_mad":      "Log montant (asymétrie corrigée)",
    "log_delta_km":         "Log distance (asymétrie corrigée)",
    "est_nuit":             "Transaction nocturne 0h-5h",
    "vitesse_km_min":       "Vitesse km/min (physiquement impossible)",
    "montant_x_etranger":   "Montant × étranger (interaction)",
    "risque_x_nb_tx":       "Risque horaire × fréquence tx",
}

X = df_model[ALL_FEATURES].values
y = df_model["fraude"].values

# Charger le meilleur modèle (gradient_boosting)
with open("models/gradient_boosting.pkl", "rb") as f:
    model = pickle.load(f)

with open("reports/models_comparison.json") as f:
    report = json.load(f)

best_model_name = report["best_model"]["name"]
print(f"   Modèle chargé : {best_model_name}")
print(f"   AUC-PR       : {report['models']['gradient_boosting']['auc_pr']}")



[1/5] Chargement des données et du modèle...
   Modèle : XGBoost
   AUC-PR : 0.8864
   Features : 17


In [ ]:
# ─────────────────────────────────────────────────────────
# APPROXIMATION SHAP VIA PERMUTATION IMPORTANCE
# (remplacez par shap.TreeExplainer quand shap est installé)
# ─────────────────────────────────────────────────────────
print("\n[2/4] Calcul des importances par permutation (≈ SHAP global)...")

# Sous-échantillon pour la rapidité
idx_sample = np.random.choice(len(X), min(2000, len(X)), replace=False)
X_sample = X[idx_sample]
y_sample = y[idx_sample]

perm_imp = permutation_importance(
    model, X_sample, y_sample,
    n_repeats=10, random_state=42,
    scoring="average_precision"
)

importances_mean = perm_imp.importances_mean
importances_std  = perm_imp.importances_std

# Top 15 features
top_idx    = np.argsort(importances_mean)[::-1][:15]
top_feats  = [ALL_FEATURES[i] for i in top_idx]
top_values = importances_mean[top_idx]
top_stds   = importances_std[top_idx]
top_labels = [FEATURE_LABELS.get(f, f) for f in top_feats]

# Figure globale
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#534AB7" if v > 0 else "#E24B4A" for v in top_values[::-1]]
bars = ax.barh(range(len(top_labels)), top_values[::-1],
               xerr=top_stds[::-1], color=colors,
               alpha=0.85, edgecolor="white", linewidth=0.5,
               capsize=3, error_kw={"linewidth": 0.8, "color": "gray"})
ax.set_yticks(range(len(top_labels)))
ax.set_yticklabels(top_labels[::-1], fontsize=9)
ax.set_title(f"Importance Globale des Features — {best_model_name}\n"
             f"(Permutation Importance sur AUC-PR · approx. SHAP values)",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Réduction de l'AUC-PR quand la feature est permutée", fontsize=9)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="both", labelsize=8)

pos_patch = mpatches.Patch(color="#534AB7", alpha=0.85, label="Impact positif sur détection")
plt.legend(handles=[pos_patch], fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("reports/shap/global_importance.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.close()
print("   → reports/shap/global_importance.png")



[2/5] Calcul des SHAP values avec TreeExplainer...
   Calcul sur 700 transactions (200 fraudes)...
   SHAP values calculées : shape=(700, 17)


In [ ]:
# ─────────────────────────────────────────────────────────
# EXPLICATIONS PAR TRANSACTION INDIVIDUELLE
# Format utilisé pour l'enregistrement blockchain
# ─────────────────────────────────────────────────────────
print("\n[3/4] Génération des explications individuelles...")

def explain_transaction(tx_row, model, feature_names, feature_labels, baseline_prob):
    """
    Approximation des SHAP values pour une transaction.
    Méthode : comparaison feature par feature avec la moyenne.
    Quand shap est installé : remplacer par shap.TreeExplainer(model).shap_values(tx_row)
    """
    tx = tx_row.reshape(1, -1)
    baseline = np.mean(tx_row.reshape(1, -1), axis=0).reshape(1, -1)
    
    prob_full = model.predict_proba(tx)[0][1]
    contributions = []
    
    for i, feat in enumerate(feature_names):
        # Masquer la feature i par sa moyenne
        tx_masked = tx.copy()
        tx_masked[0, i] = baseline[0, i]
        prob_masked = model.predict_proba(tx_masked)[0][1]
        contribution = prob_full - prob_masked
        contributions.append({
            "feature":      feat,
            "label":        feature_labels.get(feat, feat),
            "value":        round(float(tx_row[i]), 4),
            "contribution": round(float(contribution), 4),
        })
    
    # Trier par contribution absolue décroissante
    contributions.sort(key=lambda x: abs(x["contribution"]), reverse=True)
    return prob_full, contributions[:8]  # Top 8 features

# Calculer la probabilité de base (moyenne sur l'échantillon)
baseline_prob = float(np.mean(model.predict_proba(X_sample)[:, 1]))

# Sélectionner des transactions représentatives pour la démo
# 1. Transaction frauduleuse à haut score (zone rouge)
fraud_indices = np.where(y == 1)[0]
# Trouver la fraude avec le score le plus élevé
fraud_probs = model.predict_proba(X[fraud_indices])[:, 1]
best_fraud_idx = fraud_indices[np.argmax(fraud_probs)]

# 2. Transaction ambiguë (zone intermédiaire)
all_probs = model.predict_proba(X)[:, 1]
ambiguous_mask = (all_probs > 0.35) & (all_probs < 0.65)
ambiguous_indices = np.where(ambiguous_mask)[0]
if len(ambiguous_indices) > 0:
    mid_idx = ambiguous_indices[len(ambiguous_indices) // 2]
else:
    mid_idx = np.argmin(np.abs(all_probs - 0.5))

# 3. Transaction légitime (zone verte)
legit_indices = np.where(y == 0)[0]
legit_probs = model.predict_proba(X[legit_indices])[:, 1]
best_legit_idx = legit_indices[np.argmin(legit_probs)]

demo_cases = {
    "fraude_detectee":  (best_fraud_idx,   "Zone Rouge — FRAUDE automatique"),
    "cas_ambigu":       (mid_idx,          "Zone Ambiguë — révision humaine"),
    "transaction_legit":(best_legit_idx,   "Zone Verte — LÉGITIME automatique"),
}

explanations_output = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
zone_colors = {
    "fraude_detectee":   "#E24B4A",
    "cas_ambigu":        "#EF9F27",
    "transaction_legit": "#1D9E75",
}

for col_idx, (case_key, (tx_idx, case_label)) in enumerate(demo_cases.items()):
    tx_row   = X[tx_idx]
    tx_df    = df.iloc[tx_idx]
    
    score, contributions = explain_transaction(
        tx_row, model, ALL_FEATURES, FEATURE_LABELS, baseline_prob
    )
    
    zone = ("RED"    if score > 0.85 else
            "AMBER"  if score > 0.40 else "GREEN")
    
    explanation = {
        "tx_id":            str(tx_df["tx_id"]),
        "client_id":        str(tx_df["client_id"]),
        "timestamp":        str(tx_df["timestamp"]),
        "score_fraude":     round(score, 4),
        "zone":             zone,
        "decision":         ("FRAUDE" if zone == "RED" else
                             "REVUE_HUMAINE" if zone == "AMBER" else "LEGITIME"),
        "top_features":     contributions,
        "model_version":    "gradient_boosting_v1.0",
        "model_hash":       report["models"]["gradient_boosting"]["model_hash"][:16] + "...",
        "explication_hash": hashlib.sha256(
                                json.dumps(contributions, sort_keys=True).encode()
                            ).hexdigest(),
    }
    explanations_output[case_key] = explanation

    # ── Graphique pour ce cas ──────────────────────────────
    ax = axes[col_idx]
    top5 = contributions[:6]
    labels_plot = [c["label"][:28] for c in top5]
    values_plot = [c["contribution"] for c in top5]
    bar_colors  = ["#E24B4A" if v > 0 else "#1D9E75" for v in values_plot]

    ax.barh(range(len(labels_plot)), values_plot[::-1], color=bar_colors[::-1],
            alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_yticks(range(len(labels_plot)))
    ax.set_yticklabels(labels_plot[::-1], fontsize=8)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.set_title(f"{case_label}\nScore : {score:.3f}",
                 fontsize=9, fontweight="bold",
                 color=zone_colors[case_key])
    ax.set_xlabel("Contribution au score de fraude", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="both", labelsize=7)

    # Métadonnées de la transaction
    meta_text = (f"Montant : {tx_df['montant_mad']:.0f} MAD\n"
                 f"Pays    : {tx_df['pays_transaction']}\n"
                 f"Heure   : {tx_df['heure']}h\n"
                 f"Delta   : {tx_df['delta_km']:.0f} km")
    ax.text(0.98, 0.02, meta_text, transform=ax.transAxes,
            fontsize=7, va="bottom", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow",
                      edgecolor="gray", alpha=0.8))

    print(f"\n   [{case_label}]")
    print(f"     tx_id    : {explanation['tx_id']}")
    print(f"     Score    : {score:.4f}  ({zone})")
    print(f"     Décision : {explanation['decision']}")
    print(f"     Top 3 features :")
    for c in contributions[:3]:
        sign = "+" if c["contribution"] > 0 else ""
        print(f"       {sign}{c['contribution']:+.4f}  {c['label']} = {c['value']}")

plt.suptitle("Explicabilité SHAP par transaction\n(rouge = pousse vers fraude · vert = pousse vers légitime)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/shap/individual_explanations.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.close()
print("\n   → reports/shap/individual_explanations.png")



[3/5] Génération des figures SHAP globales...
   → reports/shap/01_global_importance_bar.png
   → reports/shap/02_beeswarm.png


In [ ]:
# ─────────────────────────────────────────────────────────
# SAUVEGARDE FORMAT BLOCKCHAIN
# ─────────────────────────────────────────────────────────
print("\n[4/4] Sauvegarde du format d'explication blockchain...")

blockchain_payload = {
    "model_key":       "gradient_boosting",
    "model_hash":      report["models"]["gradient_boosting"]["model_hash"],
    "global_importance": [
        {
            "rank":         i + 1,
            "feature":      ALL_FEATURES[top_idx[i]],
            "label":        FEATURE_LABELS.get(ALL_FEATURES[top_idx[i]], ALL_FEATURES[top_idx[i]]),
            "importance":   round(float(importances_mean[top_idx[i]]), 6),
        }
        for i in range(min(10, len(top_idx)))
    ],
    "demo_explanations": explanations_output,
    "explanation_format": "permutation_importance_approx",
    "note": (
        "Remplacer par shap.TreeExplainer(model).shap_values(X) "
        "après pip install shap pour des valeurs SHAP exactes. "
        "Le format JSON de sortie reste identique — seules les valeurs changent."
    ),
}

with open("reports/shap/blockchain_explanations.json", "w", encoding="utf-8") as f:
    json.dump(blockchain_payload, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("  EXPLICABILITÉ GÉNÉRÉE AVEC SUCCÈS")
print("="*60)
print("  Fichiers :")
print("    reports/shap/global_importance.png")
print("    reports/shap/individual_explanations.png")
print("    reports/shap/blockchain_explanations.json")
print("\n  Format JSON prêt pour RecordDecision() sur blockchain.")
print("  Le champ 'explication_hash' est le hash IPFS CID à stocker.")
print("="*60)



[4/5] Génération des explications individuelles...

   [Zone Rouge — FRAUDE automatique]
     tx_id    : TX00031491
     Score    : 0.9997  (RED)
     Décision : FRAUDE
     Top 3 SHAP features :
       SHAP=+2.6892  Ratio montant / moyenne client = 34.714
       SHAP=+2.0279  Montant de la transaction (MAD) = 12588.84
       SHAP=+1.5172  Heure de la transaction = 3.0

   [Zone Ambiguë — révision humaine]
     tx_id    : TX00033454
     Score    : 0.6271  (AMBER)
     Décision : REVUE_HUMAINE
     Top 3 SHAP features :
       SHAP=+2.3715  Ratio montant / moyenne client = 2.112
       SHAP=-0.2813  Ancienneté du compte (jours) = 701.0
       SHAP=-0.2490  Montant de la transaction (MAD) = 243.49

   [Zone Verte — LÉGITIME automatique]
     tx_id    : TX00027697
     Score    : 0.0010  (GREEN)
     Décision : LEGITIME
     Top 3 SHAP features :
       SHAP=-2.3550  Ratio montant / moyenne client = 1.942
       SHAP=-0.8477  Délai depuis dernière tx (min) = 216577.0
       SHAP=-0.5676